# 01 Dataset Audit

Checks local files, valid zip members, headers, label columns, and writes `results/dataset_audit.json`.

In [ ]:
from pathlib import Path
import os
import sys

ABS_PROJECT_ROOT = Path('/Volumes/LocalDrive1/onlyscholar-projects/EcoTrust-FLIDS[DeepakSir]/IMPLEMENTATION')
DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/projects/implementation/EcoTrust-FLIDS[DeepakSir]/IMPLEMENTATION')


def find_project_root():
    candidates = []
    env_root = os.environ.get('ECOTRUST_PROJECT_ROOT')
    if env_root:
        candidates.append(Path(env_root))
    candidates.extend([Path.cwd(), Path.cwd().parent, DRIVE_PROJECT_ROOT, ABS_PROJECT_ROOT])
    for base in list(candidates):
        candidates.extend(base.parents)
    seen = set()
    for candidate in candidates:
        expanded = candidate.expanduser()
        resolved = expanded.resolve() if expanded.exists() else expanded
        if str(resolved) in seen:
            continue
        seen.add(str(resolved))
        if (resolved / 'configs' / 'project.json').exists() and (resolved / 'src' / 'ecotrust').exists():
            return resolved
    raise RuntimeError(
        'Cannot find EcoTrust-FLIDS IMPLEMENTATION. In Colab, mount Drive and set ECOTRUST_PROJECT_ROOT to '
        '/content/drive/MyDrive/projects/implementation/EcoTrust-FLIDS[DeepakSir]/IMPLEMENTATION.'
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
if str(PROJECT_ROOT).startswith('/content/drive') and (PROJECT_ROOT / 'configs' / 'project_colab.json').exists():
    CONFIG_PATH = PROJECT_ROOT / 'configs' / 'project_colab.json'
else:
    CONFIG_PATH = PROJECT_ROOT / 'configs' / 'project.json'
RUN_MODE = 'full'  # change to 'full' after smoke run passes
DATASETS = ['Edge-IIoTset', 'TON_IoT', 'CICIoT2023']
print('PROJECT_ROOT:', PROJECT_ROOT)
print('CONFIG_PATH:', CONFIG_PATH)

from ecotrust.config import load_config, ensure_project_dirs
from ecotrust.data.audit import audit_datasets
cfg = load_config(CONFIG_PATH)
ensure_project_dirs(cfg)
report = audit_datasets(cfg, cfg.results_root / 'dataset_audit.json')
for name, item in report['datasets'].items():
    print(name, 'exists=', item['exists'], 'files=', len(item['files']), 'archives=', len(item['archives']))
    for check in item['preferred_source_checks']:
        print(' ', check['source'], 'label_present=', check.get('label_column_present'), 'columns=', check.get('n_columns'))